# Experiment 6: GloVe Word Embeddings & N-gram Statistical Language Model

## Lab Questions / Objectives

1. **To implement the GloVe (Global Vectors for Word Representation) algorithm to generate word embeddings from the text corpus and find most similar words of “King”.**
   * *Requirements:* Two separate codes:
     1. Without using any pretrained model (From Scratch)
     2. Using Pretrained model (Gensim)
   * *Dataset:* `glove_corpus.txt`

2. **To implement a statistical language model from scratch using the N-gram approach to predict the next word in a sequence based on the preceding words in a text corpus.**

In [2]:
# ==============================================================================
# PART 1A: GloVe Implementation from Scratch (PyTorch)
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import defaultdict
!pip install gensim
import gensim.downloader as api

# Corpus from glove_corpus.txt
corpus = [
    "The king rules the kingdom.", "The queen rules the kingdom.",
    "The king is a powerful man.", "The queen is a wise woman.",
    "The man is strong and brave.", "The woman is intelligent and kind.",
    "The king and queen rule together.", "The man and woman live in the kingdom.",
    "The prince is the son of the king.", "The princess is the daughter of the queen.",
    "The king lives in a royal palace.", "The queen lives in a royal palace.",
    "A doctor treats a patient.", "A teacher teaches students in a school.",
    "Students read books and learn new things.", "A computer processes data and information.",
    "A computer scientist develops software.", "The scientist conducts research in a laboratory.",
    "The cat drinks milk.", "The dog eats food.", "The cat and dog live in a house.",
    "The sun gives light and heat.", "The moon shines at night.", "The earth revolves around the sun."
]

def tokenize(text):
    return text.lower().replace('.', '').split()

tokens_list = [tokenize(sent) for sent in corpus]
vocab = sorted(list(set(word for sent in tokens_list for word in sent)))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)

# Build Co-occurrence Matrix
window_size = 2
co_matrix = np.zeros((vocab_size, vocab_size), dtype=np.float32)

for tokens in tokens_list:
    for i, target in enumerate(tokens):
        target_idx = word2idx[target]
        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                context_idx = word2idx[tokens[j]]
                co_matrix[target_idx, context_idx] += 1.0 / abs(i - j)

# GloVe Model Architecture
class GloVeModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=25, x_max=100, alpha=0.75):
        super(GloVeModel, self).__init__()
        self.w = nn.Embedding(vocab_size, embed_dim)
        self.w_tilde = nn.Embedding(vocab_size, embed_dim)
        self.b = nn.Embedding(vocab_size, 1)
        self.b_tilde = nn.Embedding(vocab_size, 1)
        self.x_max = x_max
        self.alpha = alpha

        nn.init.uniform_(self.w.weight, -0.5, 0.5)
        nn.init.uniform_(self.w_tilde.weight, -0.5, 0.5)
        nn.init.constant_(self.b.weight, 0.0)
        nn.init.constant_(self.b_tilde.weight, 0.0)

    def forward(self, i, j, x_ij):
        w_i = self.w(i)
        w_j = self.w_tilde(j)
        b_i = self.b(i).squeeze()
        b_j = self.b_tilde(j).squeeze()

        dot = torch.sum(w_i * w_j, dim=1)
        loss = (dot + b_i + b_j - torch.log(x_ij)) ** 2
        f_x = torch.clamp((x_ij / self.x_max) ** self.alpha, max=1.0)
        return torch.mean(f_x * loss)

i_idx, j_idx, x_vals = [], [], []
for i in range(vocab_size):
    for j in range(vocab_size):
        if co_matrix[i, j] > 0:
            i_idx.append(i)
            j_idx.append(j)
            x_vals.append(co_matrix[i, j])

i_tensor = torch.tensor(i_idx, dtype=torch.long)
j_tensor = torch.tensor(j_idx, dtype=torch.long)
x_tensor = torch.tensor(x_vals, dtype=torch.float32)

glove_model = GloVeModel(vocab_size, embed_dim=25)
optimizer = optim.Adagrad(glove_model.parameters(), lr=0.05)

for epoch in range(300):
    optimizer.zero_grad()
    loss = glove_model(i_tensor, j_tensor, x_tensor)
    loss.backward()
    optimizer.step()

final_embeds = (glove_model.w.weight + glove_model.w_tilde.weight).detach().numpy()

def get_similar_scratch(word, top_n=3):
    if word not in word2idx:
        return "Word not in vocabulary."
    w_vec = final_embeds[word2idx[word]]
    sims = np.dot(final_embeds, w_vec) / (np.linalg.norm(final_embeds, axis=1) * np.linalg.norm(w_vec))
    sorted_indices = np.argsort(-sims)
    return [(idx2word[idx], float(sims[idx])) for idx in sorted_indices[1:top_n+1]]

print("--- GloVe (From Scratch) Most Similar to 'king' ---")
print(get_similar_scratch("king"))


# ==============================================================================
# PART 1B: GloVe Using Pretrained Model (Gensim)
# ==============================================================================
print("\nLoading Pretrained GloVe (wiki-gigaword-50)...")
gensim_glove = api.load("glove-wiki-gigaword-50")

print("--- GloVe (Pretrained Gensim) Most Similar to 'king' ---")
for word, score in gensim_glove.most_similar("king", topn=5):
    print(f"{word}: {score:.4f}")


# ==============================================================================
# PART 2: Statistical Language Model using N-gram Approach
# ==============================================================================
ngram_corpus = [
    "natural language processing is a branch of artificial intelligence.",
    "natural language processing enables computers to understand human language.",
    "machine learning is a branch of artificial intelligence.",
    "deep learning is a part of machine learning.",
    "language models learn patterns from text data.",
    "a language model predicts the next word in a sentence.",
    "natural language processing uses language models.",
    "artificial intelligence includes machine learning and deep learning."
]

class BigramLanguageModel:
    def __init__(self):
        self.counts = defaultdict(lambda: defaultdict(int))
        self.context_counts = defaultdict(int)

    def train(self, corpus):
        for sentence in corpus:
            tokens = sentence.lower().replace('.', '').split()
            for w1, w2 in zip(tokens[:-1], tokens[1:]):
                self.counts[w1][w2] += 1
                self.context_counts[w1] += 1

    def predict_next(self, word):
        word = word.lower()
        if word not in self.counts or self.context_counts[word] == 0:
            return "Context word not found."

        probabilities = {
            next_w: count / self.context_counts[word]
            for next_w, count in self.counts[word].items()
        }
        best_word = max(probabilities, key=probabilities.get)
        return best_word, probabilities[best_word]

ngram_model = BigramLanguageModel()
ngram_model.train(ngram_corpus)

print("\n--- Next Word Predictions (N-gram Model) ---")
test_prompts = ["natural", "artificial", "machine", "deep"]
for context in test_prompts:
    pred_word, prob = ngram_model.predict_next(context)
    print(f"Prompt: '{context}' -> Next Word: '{pred_word}' (Prob: {prob:.2f})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 9.8 MB/s  0:00:02
--- GloVe (From Scratch) Most Similar to 'king' ---
[('man', 0.37545984983444214), ('laboratory', 0.370505154132843), ('teaches', 0.33538177609443665)]

Loading Pretrained GloVe (wiki-gigaword-50)...
[==================================================] 100.0% 66.0/66.0MB downloaded
--- GloVe (Pretrained Gensim) Most Similar to 'king' ---
prince: 0.8236
queen: 0.7839
ii: 0.7746
emperor: 0.7736
son: 0.7667

--- Next Word Predictions (N-gram Model) ---
Prompt: 'natural' -> Next Word: 'language' (Prob: 1.00)
Prompt: 'artificial' -> Next Word: 'intelligence' (Prob: 1.00)
Prompt: 'machine' -> Next Word: 'learning' (Prob: 1.00)
Prompt: 'deep' -> Next Word: 'learning' (Prob: 1.00)
